# FleetSafe GNM-VLNVerse Reproducibility Notebook
Source-code validation and reproducibility layer for the MobileNetV2-GNM
baseline, EMA ablation, W&B/MLOps tracking, Offline VLN evaluation,
dataset provenance and Isaac physics Collision Rate evidence.

**Scope guard: this documents a controlled adapted VLNVerse/VLNTube
subset and smoke evaluations — NOT a full VLNVerse benchmark.**

Run from the repo root in the `gnm_train` environment.

## 1. Git state and repo provenance

In [1]:
import subprocess, json, os
if not os.path.exists('configs/gnm/gnm_base.yaml'):
    os.chdir('..')   # nbconvert runs from notebooks/
assert os.path.exists('configs/gnm/gnm_base.yaml'), 'run from repo root'
g = lambda *a: subprocess.run(['git']+list(a), capture_output=True, text=True).stdout.strip()
print('branch :', g('branch','--show-current'))
print('commit :', g('rev-parse','--short','HEAD'))
print('dirty  :', g('status','--porcelain')[:200] or 'clean (tracked)')
print(g('log','--oneline','-5'))

branch : isaac-hospital-demo
commit : ec453ce
dirty  : ?? assets/experiments/rosbags/
?? assets/robots/yahboom_m3_pro/yahboom_m3pro.bak-20260708-0602.usd
?? assets/robots/yahboom_m3_pro/yahboom_m3pro.bak-20260708-0612.usd
?? assets/robots/yahboom_m3_pro/y
ec453ce Add professor slides for MobileNetV2 EMA ablation
c8e5642 Add final Isaac Sim live evidence stage with selected checkpoint
82e218e Add dataset provenance, protocol alignment and professor update
e7222b4 Complete MobileNetV2 EMA ablation and physics evaluation
91c9716 Run fresh held-out normalized_pred authority rerun: REJECTED


## 2. PyTorch/CUDA production-grade verification

In [2]:
import torch
print('torch:', torch.__version__)
assert torch.cuda.is_available(), 'CUDA required'
print('gpu  :', torch.cuda.get_device_name(0))
x = torch.randn(1024,1024, device='cuda'); y = x@x
print('cuda matmul ok:', tuple(y.shape), '| alloc MB:', round(torch.cuda.memory_allocated()/2**20,1))

torch: 2.11.0+cu128
gpu  : NVIDIA GeForce RTX 4080 SUPER
cuda matmul ok: (1024, 1024) | alloc MB: 16.1


## 3. Environment snapshot and dependency capture

In [3]:
import numpy, sys, subprocess
print('python:', sys.version.split()[0], '| numpy:', numpy.__version__)
n = len(subprocess.run([sys.executable,'-m','pip','freeze'], capture_output=True, text=True).stdout.splitlines())
print(f'{n} packages; full freeze: assets/experiments/training/mobilenet_ema_ablation_20260708/pip_freeze.txt')

python: 3.10.20 | numpy: 2.2.6


331 packages; full freeze: assets/experiments/training/mobilenet_ema_ablation_20260708/pip_freeze.txt


## 4. VLNVerse/VLNTube dataset manifest validation

In [4]:
import subprocess, json
r = subprocess.run(['python3','scripts/gnm/validate_dataset_manifest.py'], capture_output=True, text=True)
print(r.stdout); assert r.returncode == 0
m = json.load(open('assets/experiments/training_ablation/mnv2_ema_20260708/dataset_manifest.json'))
print('split :', m['split_summary'])
print('scenes:', m['scenes']['available'])
print('leakage:', m['leakage_check'])

manifest valid: 15 held-out episodes across 4 scenes; all paths exist; zero train/eval overlap (episode+trajectory level); evaluator count matches; summary references manifest; scene-level holdout honestly false (trajectory-level)
PASS: dataset manifest validated

split : {'train_episodes': 238, 'heldout_eval_episodes': 15, 'train_samples': 12421, 'val_samples': 817}
scenes: ['kujiale_0092', 'kujiale_0118', 'kujiale_0203', 'kujiale_0271']
leakage: {'train_eval_episode_overlap': 0, 'train_eval_trajectory_overlap': 0, 'scene_level_holdout': False, 'note': 'Four-scene setup: the same kujiale scenes appear in train and held-out splits, so holdout is TRAJECTORY-LEVEL, not scene-level. Stated honestly; scene-level holdout requires the planned data expansion.'}


## 5. Training commands (baseline / EMA 0.9999 / EMA 0.999)
Executed with `WANDB_MODE=offline`, seed 42, identical config except `training.ema_decay`.

In [5]:
print(open('assets/experiments/training/mobilenet_ema_ablation_20260708/training_commands.sh').read())

# executed in gnm_train env, WANDB_MODE=offline, PYTHONPATH=repo root
python scripts/gnm/04_train_gnm.py --cfg configs/gnm/gnm_base.yaml checkpoint.output_dir=checkpoints/ablation_mnv2_baseline wandb.name=mnv2_baseline_for_ema_ablation
python scripts/gnm/04_train_gnm.py --cfg configs/gnm/gnm_base.yaml training.ema_decay=0.9999 checkpoint.output_dir=checkpoints/ablation_mnv2_ema wandb.name=mnv2_ema_ablation
python scripts/gnm/04_train_gnm.py --cfg configs/gnm/gnm_base.yaml training.ema_decay=0.999 checkpoint.output_dir=checkpoints/ablation_mnv2_ema0999 wandb.name=mnv2_ema0999_sanity_ablation



## 6. Checkpoint SHA-256 manifest (checkpoints stay out of git)

In [6]:
import hashlib, os
for line in open('assets/experiments/training_ablation/mnv2_ema_20260708/checkpoint_manifest.sha256'):
    sha, path = line.split()
    ok = 'present' if os.path.exists(path) else 'NOT ON DISK'
    if os.path.exists(path):
        ok += ' sha ' + ('MATCH' if hashlib.sha256(open(path,'rb').read()).hexdigest()==sha else 'MISMATCH!')
    print(f'{sha[:12]}…  {path}  [{ok}]')

b7bac92ef7a9…  checkpoints/ablation_mnv2_baseline/best.pt  [present sha MATCH]
3520e3f8d230…  checkpoints/ablation_mnv2_ema/best.pt  [present sha MATCH]


3d668c031f77…  checkpoints/ablation_mnv2_ema0999/best.pt  [present sha MATCH]


## 7. Offline VLN evaluation — SR, OSR, NE, SPL (held-out 15 episodes)

In [7]:
import json
A='assets/experiments/training_ablation/mnv2_ema_20260708/'
for name, f, w in [('baseline','results_baseline.json','live model'),
                   ('EMA 0.9999','results_mobilenet_ema0.9999.json','EMA shadow'),
                   ('EMA 0.999','results_mobilenet_ema0.999.json','EMA shadow')]:
    r = json.load(open(A+f))
    print(f"{name:11s} SR={r.get('SR')} OSR={r.get('OSR')} NE={r.get('NE')} SPL={r.get('SPL')} [{w}]")
print('\nDecision: MobileNetV2 baseline KEPT (pre-registered rule);')
print('EMA 0.9999 degenerate (decay/horizon mismatch); EMA 0.999 no SR gain.')

baseline    SR=0.1333 OSR=0.4667 NE=6.1446 SPL=0.1333 [live model]
EMA 0.9999  SR=0.2 OSR=0.2 NE=6.0522 SPL=0.2 [EMA shadow]
EMA 0.999   SR=0.0667 OSR=0.3333 NE=6.9348 SPL=0.0667 [EMA shadow]

Decision: MobileNetV2 baseline KEPT (pre-registered rule);
EMA 0.9999 degenerate (decay/horizon mismatch); EMA 0.999 no SR gain.


## 8. Collision Rate separation — offline N/A, Isaac-only measured

In [8]:
print('Offline evaluator has NO physics contact signal: any CR it prints is')
print('0.000 BY CONSTRUCTION and must never be reported as measured.')
print('Policy: offline CR = N/A; measured CR comes only from Isaac PhysX')
print('chassis-contact events (wheel-floor excluded), verified by a positive')
print('control (27 contacts on SM_ReceptionDesk_01a).')

Offline evaluator has NO physics contact signal: any CR it prints is
0.000 BY CONSTRUCTION and must never be reported as measured.
Policy: offline CR = N/A; measured CR comes only from Isaac PhysX
chassis-contact events (wheel-floor excluded), verified by a positive
control (27 contacts on SM_ReceptionDesk_01a).


## 9. W&B / MLOps logging status

In [9]:
import json
w = json.load(open('assets/experiments/training/mobilenet_ema_ablation_20260708/wandb_runs.json'))
print('mode:', w['mode']); [print(' ', r) for r in w['runs']]
print('sync:', w['sync_instructions'])

mode: offline
  wandb/offline-run-20260708_161753-ik1f269e
  wandb/offline-run-20260708_162601-wj9vvlwk
  wandb/offline-run-20260708_164349-nbzvmz2h
sync: wandb sync wandb/offline-run-*  (project fleetsafe-gnm-vlnverse; names mnv2_baseline_for_ema_ablation / mnv2_ema_ablation / mnv2_ema0999_sanity_ablation)


## 10. Isaac physics evidence lookup

In [10]:
import json
s = json.load(open('assets/experiments/isaac_physics_eval/isaac_physics_smoke_20260708/summary.json'))
print('label:', s['label'])
for c in s['conditions']:
    print(f"{c['condition']:14s} NE={c['NE_m']} CR={c['collision_rate_episode']} "
          f"contacts={c['total_collisions']} path={c['total_path_m']}m [{c['weight_source']}]")
print('positive control:', s['sensor_positive_control']['result'][:80])
l = json.load(open('assets/experiments/isaac_live_final_20260708/live_episode_summary.json'))
print('live final:', l['aggregate'])

label: preliminary Isaac physics smoke test for Collision Rate and trajectory consistency — NOT a campaign-level benchmark
mnv2_baseline  NE=0.82 CR=0.0 contacts=0 path=8.97m [live model]
mnv2_ema0999   NE=0.821 CR=0.0 contacts=0 path=8.97m [EMA shadow]
positive control: PASS — 27 chassis-contact events logged against SM_ReceptionDesk_01a from step 5
live final: {'n': 3, 'SR': 1.0, 'OSR': 1.0, 'NE_m': 0.893, 'SPL': 0.772, 'collision_rate_episode': 0.0, 'total_chassis_contacts': 0, 'collisions_per_meter': 0.0, 'total_path_m': 11.56}


## 11. Validation target execution

In [11]:
import subprocess
for t in ['validate-dataset-manifest','validate-protocol-alignment','validate-training-mlops']:
    r = subprocess.run(['make', t], capture_output=True, text=True)
    print(t, '->', 'PASS' if r.returncode==0 else 'FAIL')
    assert r.returncode == 0, r.stdout + r.stderr

validate-dataset-manifest -> PASS
validate-protocol-alignment -> PASS
validate-training-mlops -> PASS


## 12. Claim-boundary checklist

In [12]:
checks = {
 'EMA improvement claimed': False,
 'full VLNVerse benchmark claimed': False,
 'robust navigation claimed': False,
 'language-instruction VLN claimed': False,
 'CR reported from offline evaluator': False,
 'baseline kept': True,
 'trajectory-level holdout stated honestly': True,
 '15-episode preliminary caveat stated': True,
}
for k, v in checks.items(): print(f'[{"x" if v else " "}] {k}')
s = open('docs/experiments/PROJECT_BASELINE_STATUS.md').read().lower()
assert 'fleetsafe improves' not in s and 'campaign complete' not in s
print('status-doc claim guard: OK')

[ ] EMA improvement claimed
[ ] full VLNVerse benchmark claimed
[ ] robust navigation claimed
[ ] language-instruction VLN claimed
[ ] CR reported from offline evaluator
[x] baseline kept
[x] trajectory-level holdout stated honestly
[x] 15-episode preliminary caveat stated
status-doc claim guard: OK


## 13. Professor-ready summary

In [13]:
print(open('assets/experiments/training_ablation/mnv2_ema_20260708/professor_update_onepager.md').read()[:1200])
print('… full one-pager + slides (professor_slides_mnv2_ema.pptx) in the same folder.')

# MobileNetV2-GNM Training Ablation + Isaac Physics Evaluation
*(one-page professor update — 2026-07-08)*

## 1. Objective
Test whether Exponential Moving Average (EMA) improves the current
MobileNetV2-GNM baseline, and add a real Collision Rate from Isaac Sim.
Controlled design: same four-scene VLNVerse/VLNTube split (documented in
`dataset_manifest.json`), same seed/config/evaluator — EMA the only change.

## 2. Offline VLN evaluation (held-out 15 episodes)

| Model | EMA decay | SR ↑ | OSR ↑ | NE ↓ | SPL ↑ | Collision Rate | Weights |
|---|---:|---:|---:|---:|---:|---|---|
| MobileNetV2 baseline | none | **13.3** | **46.7** | **6.14** | **0.133** | N/A — offline evaluator has no physics/contact signal | live model |
| + EMA (requested ablation) | 0.9999 | 20.0* | 20.0 | 6.05 | 0.200* | N/A — offline evaluator | EMA shadow |
| + EMA (decay-horizon sanity) | 0.999 | 6.7 | 33.3 | 6.93 | 0.067 | N/A — offline evaluator | EMA shadow |

\* Degenerate: at decay 0.9999 the shadow lags near 

## 14. Final artifact hygiene checks

In [14]:
import subprocess, re
tracked = subprocess.run(['git','ls-files'], capture_output=True, text=True).stdout
bad = [l for l in tracked.splitlines() if re.search(r'\.(pth|ckpt|db3|mcap|mp4)$', l)]
bad += [l for l in tracked.splitlines() if l.startswith(('checkpoints/','wandb/'))]
print('tracked heavy artifacts:', bad or 'none')
assert not bad
print('hygiene OK: checkpoints/rosbags/wandb/videos are out of git; manifests committed')

tracked heavy artifacts: none
hygiene OK: checkpoints/rosbags/wandb/videos are out of git; manifests committed
